In [67]:
import numpy as np
import os

print(os.listdir())

['computational_grid_gmsh_visualized.py', '.ipynb_checkpoints', 'solver_finite_volume.ipynb', 'Untitled1.ipynb', 'Untitled.ipynb', 'untitled', 'computational_grid_gmsh_visualized.ipynb', 'fluid_mesh_3d.msh', 'read_mesh_from_the_generated_mesh.ipynb']


In [68]:
import meshio

mesh = meshio.read("fluid_mesh_3d.msh")

print(mesh)


<meshio mesh object>
  Number of points: 8066
  Number of cells:
    triangle: 3068
    triangle: 66
    triangle: 132
    triangle: 132
    triangle: 130
    triangle: 132
    triangle: 66
    tetra: 44172
  Cell sets: inlet, outlet, walls, object, internalfield, gmsh:bounding_entities
  Point data: gmsh:dim_tags
  Cell data: gmsh:physical, gmsh:geometrical
  Field data: inlet, outlet, walls, object, internalfield


In [69]:
print(mesh.field_data)

{'inlet': array([1, 2]), 'outlet': array([2, 2]), 'walls': array([3, 2]), 'object': array([4, 2]), 'internalfield': array([5, 3])}


In [70]:
tetra = mesh.cells_dict["tetra"]

print(tetra.shape)

(44172, 4)


In [71]:
points = mesh.points

print(points.shape)

(8066, 3)


In [72]:
points = mesh.points
tetra = mesh.cells_dict["tetra"]

print(points.shape)
print(tetra.shape)

(8066, 3)
(44172, 4)


In [73]:
cell_centroids = np.mean(points[tetra], axis=1)

print(cell_centroids.shape)
print(cell_centroids[:5])

(44172, 3)
[[2.86158419 2.9832207  2.66361277]
 [3.28602002 2.92351644 2.53935959]
 [4.39916806 4.39554345 2.80301413]
 [2.63493961 2.83162487 2.29139132]
 [2.6548107  2.55029604 3.08550767]]


In [74]:
def compute_tetra_volumes(points, tetra):

    p1 = points[tetra[:,0]]
    p2 = points[tetra[:,1]]
    p3 = points[tetra[:,2]]
    p4 = points[tetra[:,3]]

    volumes = np.abs(
        np.einsum(
            'ij,ij->i',
            p2-p1,
            np.cross(p3-p1, p4-p1)
        )
    ) / 6.0

    return volumes

In [75]:
cell_volumes = compute_tetra_volumes(points, tetra)

print(cell_volumes.shape)
print(cell_volumes[:10])
print("Minimum volume :", np.min(cell_volumes))
print("Maximum volume :", np.max(cell_volumes))

(44172,)
[3.79798134e-05 1.92538443e-05 1.36791468e-01 5.72642216e-05
 5.00828504e-05 1.08661936e-03 2.13798471e-05 1.43500528e-02
 4.37836868e-05 2.05458110e-05]
Minimum volume : 6.560122315883108e-06
Maximum volume : 0.5210590934771592


In [76]:
print("Total mesh volume =", np.sum(cell_volumes))

Total mesh volume = 249.4783003021887


In [77]:
import numpy as np

def extract_faces(tetra):
    """
    Extract all triangular faces from a tetrahedral mesh.

    Parameters
    ----------
    tetra : ndarray of shape (n_cells, 4)
        Tetrahedral connectivity array.

    Returns
    -------
    faces : ndarray of shape (4*n_cells, 3)
        Sorted node indices for every triangular face.

    face_owner : ndarray of shape (4*n_cells,)
        Cell index that owns each extracted face.
    """

    n_cells = tetra.shape[0]

    faces = np.empty((4 * n_cells, 3), dtype=np.int64)
    face_owner = np.empty(4 * n_cells, dtype=np.int64)

    k = 0

    for cell_id, cell in enumerate(tetra):

        n0, n1, n2, n3 = cell

        cell_faces = [
            (n0, n1, n2),
            (n0, n1, n3),
            (n0, n2, n3),
            (n1, n2, n3)
        ]

        for face in cell_faces:
            faces[k] = np.sort(face)
            face_owner[k] = cell_id
            k += 1

    return faces, face_owner

In [78]:
faces, face_owner = extract_faces(tetra)

print("Number of extracted faces:", len(faces))
print("Shape:", faces.shape)

print("\nFirst five faces")
print(faces[:5])

print("\nOwner cells")
print(face_owner[:5])

Number of extracted faces: 176688
Shape: (176688, 3)

First five faces
[[ 825 1867 1868]
 [ 826 1867 1868]
 [ 825  826 1867]
 [ 825  826 1868]
 [ 620  724 1869]]

Owner cells
[0 0 0 0 1]


In [79]:
def build_face_connectivity(faces, face_owner):
    """
    Build owner-neighbour connectivity from extracted faces.

    Parameters
    ----------
    faces : ndarray (n_faces,3)
    face_owner : ndarray (n_faces,)

    Returns
    -------
    unique_faces
    owner
    neighbour
    """

    face_dict = {}

    unique_faces = []
    owner = []
    neighbour = []

    for i, face in enumerate(faces):

        key = tuple(face)

        if key not in face_dict:

            face_dict[key] = len(unique_faces)

            unique_faces.append(face)
            owner.append(face_owner[i])
            neighbour.append(-1)

        else:

            idx = face_dict[key]
            neighbour[idx] = face_owner[i]

    return (
        np.asarray(unique_faces),
        np.asarray(owner),
        np.asarray(neighbour),
    )

In [80]:
unique_faces, owner, neighbour = build_face_connectivity(
    faces,
    face_owner
)

print("Unique faces :", len(unique_faces))

print("Internal faces :", np.sum(neighbour != -1))

print("Boundary faces :", np.sum(neighbour == -1))

Unique faces : 90207
Internal faces : 86481
Boundary faces : 3726


In [81]:
def compute_face_centroids(points, unique_faces):
    """
    Compute centroid of every triangular face.

    Parameters
    ----------
    points : ndarray (n_points,3)
    unique_faces : ndarray (n_faces,3)

    Returns
    -------
    face_centroids : ndarray (n_faces,3)
    """

    face_centroids = np.mean(points[unique_faces], axis=1)

    return face_centroids

In [82]:
face_centroids = compute_face_centroids(points, unique_faces)

print(face_centroids.shape)
print(face_centroids[:5])

(90207, 3)
[[2.86310469 2.98951945 2.6787658 ]
 [2.86042886 2.99358676 2.6630003 ]
 [2.84853284 2.97411587 2.65747771]
 [2.87427035 2.97566071 2.6552073 ]
 [3.28008505 2.93300517 2.53903942]]


In [83]:
def compute_face_geometry(points, unique_faces):
    """
    Compute geometric quantities for triangular faces.

    Parameters
    ----------
    points : ndarray (n_points,3)
    unique_faces : ndarray (n_faces,3)

    Returns
    -------
    face_area_vectors : ndarray (n_faces,3)
    face_areas        : ndarray (n_faces,)
    face_normals      : ndarray (n_faces,3)
    """

    p1 = points[unique_faces[:,0]]
    p2 = points[unique_faces[:,1]]
    p3 = points[unique_faces[:,2]]

    v1 = p2 - p1
    v2 = p3 - p1

    face_area_vectors = 0.5 * np.cross(v1, v2)

    face_areas = np.linalg.norm(face_area_vectors, axis=1)

    face_normals = (
        face_area_vectors /
        face_areas[:, None]
    )

    return (
        face_area_vectors,
        face_areas,
        face_normals
    )

In [84]:
face_area_vectors, face_areas, face_normals = compute_face_geometry(
    points,
    unique_faces
)

print(face_areas.shape)
print(face_normals.shape)

print(face_areas[:5])
print(face_normals[:5])

(90207,)
(90207, 3)
[0.00231408 0.00312412 0.00177744 0.00173394 0.00136398]
[[-0.1025557   0.32892444 -0.93877103]
 [-0.09774477  0.83685734 -0.53862395]
 [ 0.79574673  0.60526219  0.02109094]
 [ 0.85495441 -0.44837944 -0.26078503]
 [-0.68394383  0.69538856  0.22058011]]


In [85]:
def orient_face_normals(face_centroids,
                        cell_centroids,
                        owner,
                        face_normals,
                        face_area_vectors):

    face_normals = face_normals.copy()
    face_area_vectors = face_area_vectors.copy()

    for i in range(len(owner)):

        owner_cell = owner[i]

        d = face_centroids[i] - cell_centroids[owner_cell]

        if np.dot(face_normals[i], d) < 0:

            face_normals[i] *= -1
            face_area_vectors[i] *= -1

    return face_normals, face_area_vectors

In [86]:
face_normals, face_area_vectors = orient_face_normals(
    face_centroids,
    cell_centroids,
    owner,
    face_normals,
    face_area_vectors
)

In [87]:
for i, cell_block in enumerate(mesh.cells):
    print(i, cell_block.type, len(cell_block.data))

0 triangle 3068
1 triangle 66
2 triangle 132
3 triangle 132
4 triangle 130
5 triangle 132
6 triangle 66
7 tetra 44172


In [88]:
for i, data in enumerate(mesh.cell_data["gmsh:physical"]):
    print(i, np.unique(data))

0 [4]
1 [1]
2 [3]
3 [3]
4 [3]
5 [3]
6 [2]
7 [5]


In [89]:
boundary_faces = {
    "inlet": ...,
    "outlet": ...,
    "walls": ...,
    "object": ...
}

In [90]:
def create_face_lookup(unique_faces):
    """
    Create a dictionary that maps a sorted face (node tuple)
    to its index in unique_faces.
    """

    lookup = {}

    for i, face in enumerate(unique_faces):
        lookup[tuple(face)] = i

    return lookup

In [91]:
face_lookup = create_face_lookup(unique_faces)

print(len(face_lookup))

90207


In [92]:
def classify_boundary_faces(mesh, face_lookup):
    """
    Map boundary triangles from Gmsh to the corresponding
    face indices in the finite-volume mesh.
    """

    # Physical tags
    physical_names = {
        1: "inlet",
        2: "outlet",
        3: "walls",
        4: "object"
    }
def classify_boundary_faces(mesh, face_lookup):
    """
    Map boundary triangles from Gmsh to the corresponding
    face indices in the finite-volume mesh.
    """

    # Physical tags
    physical_names = {
        1: "inlet",
        2: "outlet",
        3: "walls",
        4: "object"
    }

    boundary_faces = {
        "inlet": [],
        "outlet": [],
        "walls": [],
        "object": []
    }

    # Ignore the last block (tetrahedra)
    for block, tags in zip(mesh.cells[:-1],
                           mesh.cell_data["gmsh:physical"][:-1]):

        triangles = block.data

        for tri, tag in zip(triangles, tags):

            key = tuple(sorted(tri))

            face_index = face_lookup[key]

            boundary_faces[physical_names[int(tag)]].append(face_index)

    return boundary_faces
    boundary_faces = {
        "inlet": [],
        "outlet": [],
        "walls": [],
        "object": []
    }

    # Ignore the last block (tetrahedra)
    for block, tags in zip(mesh.cells[:-1],
                           mesh.cell_data["gmsh:physical"][:-1]):

        triangles = block.data

        for tri, tag in zip(triangles, tags):

            key = tuple(sorted(tri))

            face_index = face_lookup[key]

            boundary_faces[physical_names[int(tag)]].append(face_index)

    return boundary_faces

In [93]:
boundary_faces = classify_boundary_faces(mesh, face_lookup)

for name in boundary_faces:
    print(name, len(boundary_faces[name]))

inlet 66
outlet 66
walls 526
object 3068


In [94]:
import numpy as np

np.savez(
    "processed_mesh.npz",

    points=points,
    tetra=tetra,

    cell_centroids=cell_centroids,
    cell_volumes=cell_volumes,

    unique_faces=unique_faces,

    owner=owner,
    neighbour=neighbour,

    face_centroids=face_centroids,

    face_area_vectors=face_area_vectors,
    face_areas=face_areas,
    face_normals=face_normals
)

print("Processed mesh saved successfully.")

Processed mesh saved successfully.
